In [76]:
import pandas as pd
import numpy as np
import sqlite3
import json

np.random.seed(42)
n = 800

# --- Full Region → Cities Mapping ---
region_city_map = {
    'North': ['Colombo', 'Kolonnawa', 'Aluthkade', 'Colpetty', 'Union Place'],
    'South': ['Moratuwa', 'Kesbewa', 'Piliyandala', 'Polgasowita', 'Ratmalana'],
    'East': ['Malabe', 'Kotte', 'Avissawella', 'Padukka', 'Kottawa'],
    'West': ['Bambalapitiya', 'Wellawatthe', 'Pamankada', 'Dehiwala', 'Havelock', 'Thimbirigasyaya']
}

# --- 1. Demographics ---
income_skewed = np.random.lognormal(mean=10, sigma=0.5, size=n)
income_skewed = np.round(income_skewed, -3)

df_demo = pd.DataFrame({
    'Customer_ID': range(1, n+1),
    'Age': np.random.randint(18, 70, size=n),
    'Income': income_skewed,
    'Gender': np.random.choice(['Male', 'Female'], size=n),
    'Education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], size=n)
})

# --- Add City to Demographics (from full map) ---
all_cities = [city for cities in region_city_map.values() for city in cities]
df_demo['City'] = np.random.choice(all_cities, size=n)

# Number of high-income outliers
n_outliers = 3

# Randomly choose indices for outliers
outlier_indices = np.random.choice(df_demo.index, size=n_outliers, replace=False)

# Assign very high income values (in millions)
df_demo.loc[outlier_indices, 'Income'] = np.round(np.random.randint(1_000_000, 5_000_000, size=n_outliers),-3)


In [77]:
df_demo

,Customer_ID,Age,Income,Gender,Education,City
0,1,35,28000.0,Male,High School,Kotte
1,2,59,21000.0,Male,Bachelor,Piliyandala
2,3,56,30000.0,Male,PhD,Piliyandala
3,4,34,47000.0,Female,Master,Thimbirigasyaya
4,5,31,20000.0,Male,High School,Avissawella
...,...,...,...,...,...,...
795,796,33,37000.0,Female,High School,Bambalapitiya
796,797,64,22000.0,Male,High School,Kottawa
797,798,32,31000.0,Female,PhD,Colpetty
798,799,42,22000.0,Male,PhD,Kesbewa


In [78]:
import sqlite3

# --- Connect to SQLite (or create new database) ---
conn = sqlite3.connect('customers_demo.db')

# --- Save the demographics table ---
df_demo.to_sql('demographics', conn, index=False, if_exists='replace')

# --- Optional: Verify by reading back ---
df_check = pd.read_sql('SELECT * FROM demographics LIMIT 10', conn)
print(df_check)

# --- Close the connection ---
conn.close()


   Customer_ID  Age   Income  Gender    Education             City
0            1   35  28000.0    Male  High School            Kotte
1            2   59  21000.0    Male     Bachelor      Piliyandala
2            3   56  30000.0    Male          PhD      Piliyandala
3            4   34  47000.0  Female       Master  Thimbirigasyaya
4            5   31  20000.0    Male  High School      Avissawella
5            6   48  20000.0    Male     Bachelor            Kotte
6            7   41  49000.0  Female     Bachelor    Bambalapitiya
7            8   52  32000.0    Male     Bachelor          Colombo
8            9   61  17000.0  Female          PhD      Avissawella
9           10   62  29000.0  Female          PhD          Kesbewa


In [79]:
# --- 2. Create Location Lookup Table (City → Region) ---
location_data = []
for region, cities in region_city_map.items():
    for city in cities:
        location_data.append({'City': city, 'Region': region})

df_location = pd.DataFrame(location_data)

# Optional: shuffle rows
df_location = df_location.sample(frac=1).reset_index(drop=True)

In [80]:
df_location

,City,Region
0,Havelock,West
1,Thimbirigasyaya,West
2,Union Place,North
3,Wellawatthe,West
4,Moratuwa,South
5,Polgasowita,South
6,Colpetty,North
7,Pamankada,West
8,Dehiwala,West
9,Kolonnawa,North


In [81]:
# Export to CSV
df_demo.to_csv('customers_demo.csv', index=False)
df_location.to_csv('colombo_locations.csv', index=False)

In [ ]:
# --- 3. Signup ---


# Step 1: create the full DataFrame with random Signup_Channel
df_signup = pd.DataFrame({
    'Customer_ID': range(1, n+1),
    'Signup_Date': pd.to_datetime('2023-01-01') + pd.to_timedelta(
        np.random.randint(0, 150, size=n), unit='d'
    ),
    'Signup_Channel': np.random.choice(['Online', 'Offline', 'Referral'], size=n)
})

# Step 2: define hardcoded values
hardcoded = {
    60:'Online', 73:'Online', 265:'Online', 327:'Online', 575:'Online',
    594:'Online', 600:'Online', 626:'Online', 663:'Online', 673:'Online',
    693:'Referral', 733:'Online', 739:'Online', 33:'Online', 218:'Online',
    224:'Online', 306:'Online', 373:'Online', 416:'Online', 428:'Referral',
    526:'Online', 554:'Online', 565:'Online', 571:'Referral', 620:'Referral',
    703:'Referral', 761:'Online', 769:'Offline', 12:'Online', 63:'Online',
    108:'Online', 107:'Offline', 114:'Online', 118:'Offline', 229:'Offline',
    332:'Online', 438:'Online', 506:'Online', 519:'Offline', 609:'Offline',
    610:'Online', 632:'Referral', 685:'Referral', 748:'Online', 750:'Referral',
    67:'Referral', 121:'Offline', 171:'Online', 257:'Referral', 269:'Online',
    350:'Online', 414:'Referral', 433:'Offline', 450:'Offline', 454:'Offline',
    518:'Offline', 618:'Referral', 638:'Offline', 643:'Offline', 649:'Offline',
    658:'Offline', 661:'Offline', 678:'Offline', 704:'Referral', 789:'Offline',
    99:'Offline', 176:'Offline', 206:'Offline', 286:'Offline', 398:'Referral',
    478:'Offline', 486:'Referral', 499:'Offline', 520:'Referral', 533:'Offline',
    534:'Offline', 633:'Offline', 46:'Offline', 94:'Offline', 276:'Offline',
    304:'Offline', 424:'Offline', 656:'Offline'
}

# Step 3: overwrite the random Signup_Channel with hardcoded values
df_signup.loc[df_signup['Customer_ID'].isin(hardcoded.keys()), 'Signup_Channel'] = \
    df_signup['Customer_ID'].map(hardcoded)

# Missing & outliers
df_signup.loc[df_signup.sample(frac=0.03).index, 'Signup_Channel'] = np.nan
df_signup.loc[np.random.choice(df_signup.index, size=3, replace=False), 'Signup_Date'] = pd.to_datetime('2030-01-01')

df_signup.to_csv('customers_signup.csv', index=False)

In [83]:
df_signup

,Customer_ID,Signup_Date,Signup_Channel
0,1,2023-05-18,Referral
1,2,2023-02-21,Referral
2,3,2023-01-15,NaN
3,4,2023-03-25,Referral
4,5,2023-05-11,Online
...,...,...,...
795,796,2023-03-06,Referral
796,797,2023-04-09,Offline
797,798,2023-03-13,Offline
798,799,2023-02-21,Offline


In [84]:
# --- 4. Churn ---
df_churn = pd.DataFrame({
    'Customer_ID': range(1, n+1),
    'Churned': np.random.choice([0, 1], size=n, p=[0.6, 0.4])
})

# Add rare invalid churn values
df_churn.loc[np.random.choice(df_churn.index, size=3, replace=False), 'Churned'] = 2

df_churn.to_csv('churn_data.csv', index=False)